[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/monacofj/misda/blob/main/examples/comparison.ipynb)

# MISDA and PCA on controlled diagnostic truth

This notebook compares MISDA and PCA on clean controlled diagnostics with explicit ground truth. The methods do not estimate the same objects: MISDA reports latent and structural dimensions while PCA provides a linear component representation. We therefore keep their native estimands separate and use a common external reconstruction score only where direct comparison is meaningful.

In [ ]:
from pathlib import Path
import subprocess
import sys

target = ".[benchmarks]" if Path("pyproject.toml").exists() else "git+https://github.com/monacofj/misda.git@main#egg=misda[benchmarks]"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", target])


In [ ]:
import pandas as pd
import misda
import misda.benchmarks as bench

N = 300
SEED = 123
COMPARISON_PROBLEM_IDS = (
    "total_redundancy",
    "blocks_4x5",
    "antagonistic_linear_groups",
    "nonlinear_blocks_4x5",
    "transitive_chain",
)

## Comparison protocol

All problems use exact observation (`Y=Z`). MISDA is evaluated against declared latent/structural truth. PCA is kept as a reconstruction curve: this notebook deliberately does **not** impose an explained-variance cutoff or other arbitrary component-selection rule. Direct comparison uses leave-one-out `global_standardized_external_r2` at the MISDA-selected dimension and also reports PCA reconstruction at the declared latent and structural dimensions.

In [ ]:
comparison_results = {}
comparison_rows = []

def curve_value(curve, dimension):
    return next(
        point[bench.COMMON_RECONSTRUCTION_METRIC]
        for point in curve
        if point["dimension"] == int(dimension)
    )

for problem_id in COMPARISON_PROBLEM_IDS:
    problem = bench.PROBLEM_BY_ID[problem_id]
    dataset = problem.generate(N=N, seed=SEED, sigma=0.0)
    truth = bench.diagnostic_truth(problem, dataset.Z)
    mis_set = misda.discover(dataset.Y, name=truth["name"], seed=SEED)
    structural = misda.rank(mis_set)
    misda.evaluate(mis_set, metrics=("linear",), candidates=structural[:1])
    benchmark_result = misda.benchmark(mis_set, truth)
    print(benchmark_result.report())
    mis_set.graph_plot(ranking=structural)

    pca_external_curve = bench.pca_external_reconstruction_curve(
        dataset.Y, max_components=dataset.Y.shape[1]
    )
    pca_native_curve = bench.pca_in_sample_reconstruction_curve(
        dataset.Y, max_components=dataset.Y.shape[1]
    )
    misda_common = bench.misda_global_standardized_external_r2(dataset.Y, mis_set)
    selected_dimension = structural.selected_dimension
    latent_truth = truth["latent_expected"]
    structural_truth = truth["structural_expected"]

    comparison_results[problem_id] = {
        "dataset": dataset,
        "truth": truth,
        "result_obj": mis_set,
        "ranking": structural,
        "benchmark_obj": benchmark_result,
        "pca_external_curve": pca_external_curve,
        "pca_native_curve": pca_native_curve,
    }
    comparison_rows.append({
        "problem_id": problem_id,
        "name": truth["name"],
        "latent_truth": latent_truth,
        "structural_truth": structural_truth,
        "misda_latent": mis_set.analysis.latent_dimension,
        "misda_structural": mis_set.analysis.structural_dimension,
        "misda_selected": selected_dimension,
        "misda_latent_error": abs(mis_set.analysis.latent_dimension - latent_truth),
        "misda_structural_error": abs(mis_set.analysis.structural_dimension - structural_truth),
        "misda_global_standardized_external_r2": misda_common,
        "pca_at_misda_dimension": curve_value(pca_external_curve, selected_dimension),
        "pca_at_latent_truth": curve_value(pca_external_curve, latent_truth),
        "pca_at_structural_truth": curve_value(pca_external_curve, structural_truth),
    })

comparison = pd.DataFrame(comparison_rows)

In [ ]:
comparison

## Interpretation

`misda_latent_error` and `misda_structural_error` are genuine errors against MISDA's declared estimands. There is intentionally no `pca_dimension_error`: PCA has not been given a component-selection rule, and choosing one here would introduce an external hyperparameter. The PCA columns instead answer reconstruction questions at fixed, explicitly named dimensions. The antagonistic two-group diagnostic is especially useful: its latent truth is 1 while its structural truth is 2, making the distinction between latent dependence and positive-redundancy structure visible without forcing PCA components to stand in for both.

In [ ]:
pca_reconstruction = pd.DataFrame(
    {
        "problem_id": problem_id,
        "dimension": point["dimension"],
        "global_standardized_external_r2": point[bench.COMMON_RECONSTRUCTION_METRIC],
    }
    for problem_id, item in comparison_results.items()
    for point in item["pca_external_curve"]
)
pca_reconstruction